<a href="https://colab.research.google.com/github/ansonkwokth/TableTennisPrediction/blob/dev/NNs_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!git clone https://github.com/ansonkwokth/TableTennisPrediction.git
%cd TableTennisPrediction

Cloning into 'TableTennisPrediction'...
remote: Enumerating objects: 370, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 370 (delta 34), reused 15 (delta 15), pack-reused 311 (from 2)
Receiving objects: 100% (370/370), 6.78 MiB | 30.06 MiB/s, done.
Resolving deltas: 100% (180/180), done.
/content/TableTennisPrediction/TableTennisPrediction


In [34]:
import pandas as pd
from utils import data_loader as dl

import numpy as np
from model.Elo import Elo
from model.ModifiedElo import ModifiedElo
from model.ensemble import BaggingRatingSystem

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

import copy
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

# Data

In [35]:
# GAME = 'TTStar'
# GAME = 'TTCup'

# GAME = 'SetkaCup'
GAME = 'SetkaCupWomen'
# GAME = 'LigaPro'


In [36]:
match GAME:
    case 'TTStar':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'TTCup':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'SetkaCup':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'SetkaCupWomen':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'LigaPro':
        years = [2022, 2023, 2024]
    case _:
        raise ValueError("Invalid game selected.")


text_data_game = dl.load_game_data(GAME, years, '../')
text_data = {
    year: text_data_game[year] for year in years
}
df = dl.create_game_dfs(GAME, years, text_data)

Loading ..//SetkaCupWomen2020.txt
Loading ..//SetkaCupWomen2021.txt
Loading ..//SetkaCupWomen2022.txt
Loading ..//SetkaCupWomen2023.txt
Loading ..//SetkaCupWomen2024.txt


In [37]:
# Generate ID indices for each pair of rows in the DataFrame
idx_lt = [i for i in range(len(df) // 2) for _ in range(2)]
df['ID'] = idx_lt  # Assign to the 'ID' column

# Reset the DataFrame index to ensure it's sequential
df.reset_index(drop=True, inplace=True)

# Get unique players and store them in player_lt
player_lt = df['Player'].unique()



In [38]:
year_val = years[-2]
year_test = years[-1]

df_train = df.loc[pd.DatetimeIndex(df['Date']).year < year_val]
df_val = df.loc[pd.DatetimeIndex(df['Date']).year == year_val]
df_train_val = df.loc[pd.DatetimeIndex(df['Date']).year <= year_val]
df_test = df.loc[pd.DatetimeIndex(df['Date']).year == year_test]

In [39]:
def format_to_array(df: pd.DataFrame) -> np.ndarray:

    # info_col = ['ID', 'Round', 'Datetime', 'Game', 'Date', 'Time']
    info_col = ['Round', 'Datetime', 'Game', 'Date', 'Time']
    col = [item for item in df.columns if item not in info_col]

    df[[c for c in col if "Set" in c]] = df[[c for c in col if "Set" in c]].astype(float)
    X = df[col].values.reshape(-1, 2, len(col))
    return X

In [40]:
X_train = format_to_array(df_train)
X_train_val = format_to_array(df_train_val)
X_val = format_to_array(df_val)
X_test = format_to_array(df_test)

In [41]:
X_all = format_to_array(df)

In [42]:
# def get_data(X):
#     data_all = []
#     for game in X:
#         game = game[:, 1:]
#         player1, player2 = game[:, 0]
#         scores = game[:, 1:]

#         win1 = sum(scores[0]>scores[1])
#         win2 = sum(scores[0]<scores[1])
#         p1_win = int(win1 > win2)

#         for si in scores.T:
#             if (si[0] + si[1]) == 0: continue
#             ti = si[0] / (si[0] + si[1])
#             di = si[0] - si[1]
#             if not np.isnan(ti):
#                 data_all.append([player1, player2, ti, di, p1_win])
#     return data_all

# def get_data_idx(data, player_to_idx):
#     data_all_idx = []
#     for game in data:
#         data_all_idx.append((player_to_idx[game[0]], player_to_idx[game[1]], game[2], game[3] , game[4]))
#     return data_all_idx

# data_all = get_data(X_all)
# data_train = get_data(X_train)
# data_train_val = get_data(X_train_val)
# data_test = get_data(X_test)
# player_to_idx = {pn: i for i, pn in enumerate(np.unique(np.array(data_all)[:, :2]))}


# data_all_idx = get_data_idx(data_all, player_to_idx)
# data_train_idx = get_data_idx(data_train, player_to_idx)
# data_train_val_idx = get_data_idx(data_train_val, player_to_idx)
# data_test_idx = get_data_idx(data_test, player_to_idx)

In [54]:

player_to_idx = {pn: i for i, pn in enumerate(np.unique(np.array(X_train)[:, :, 1].reshape(-1)))}

In [95]:
def get_data(X):
    data_all = []
    for game in X:
        game = game[:, 1:]
        print(game)

        player1, player2 = game[:, 0]
        scores = game[:, 1:].astype(float)

        win1 = sum(scores[0]>scores[1])
        win2 = sum(scores[0]<scores[1])
        p1_win = int(win1 > win2)
        print(p1_win)
        print(scores)
        t = scores[0] / (scores[0] + scores[1])

        mask = (1-np.isnan(t)).astype(int)
        print(t, mask)
        dfs
    #     for si in scores.T:
    #         if (si[0] + si[1]) == 0: continue
    #         ti = si[0] / (si[0] + si[1])
    #         di = si[0] - si[1]
    #         if not np.isnan(ti):
    #             data_all.append([player1, player2, ti, di, p1_win])
    # return data_all

In [1]:

get_data(X_train)

NameError: name 'get_data' is not defined

In [26]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader



class TableTennisDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        p1, p2, t, d, pb = self.data[idx]
        return int(p1), int(p2), t, d, pb


dataset_train = TableTennisDataset(data_train_idx)
dataset_train_val = TableTennisDataset(data_train_val_idx)
dataloader_train = DataLoader(dataset_train, batch_size=32, shuffle=True)
dataloader_train_val = DataLoader(dataset_train_val, batch_size=32, shuffle=True)

